# QML Basics - PennyLane + Qiskit

Building a variational classifier on toy data, then a quantum kernel,
then running the same circuit through Qiskit.

In [1]:
%pip install pennylane pennylane-qiskit qiskit qiskit-aer scikit-learn torch

import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

print(qml.version())

   ---------------------------------------- 0.0/8.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.6 MB 1.8 MB/s eta 0:00:05
   ---- ----------------------------------- 1.0/8.6 MB 2.0 MB/s eta 0:00:04
   ------- -------------------------------- 1.6/8.6 MB 2.1 MB/s eta 0:00:04
   --------- ------------------------------ 2.1/8.6 MB 2.2 MB/s eta 0:00:04
   ------------ --------------------------- 2.6/8.6 MB 2.2 MB/s eta 0:00:03
   -------------- ------------------------- 3.1/8.6 MB 2.3 MB/s eta 0:00:03
   ----------------- ---------------------- 3.7/8.6 MB 2.3 MB/s eta 0:00:03
   ------------------- -------------------- 4.2/8.6 MB 2.3 MB/s eta 0:00:02
   --------------------- ------------------ 4.7/8.6 MB 2.4 MB/s eta 0:00:02
   ------------------------ --------------- 5.2/8.6 MB 2.4 MB/s eta 0:00:02
   ------------------------- -------------- 5.5/8.6 MB 2.3 MB/s eta 0:00:02
   -----------------------


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


0.45.1


## 1. Data

Using make_moons since it's not linearly separable. Scaling features to [0, pi] so I can
plug them straight in as rotation angles. Labels are -1/+1 to match <Z>, which lives in [-1, 1].

In [2]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

X, y = make_moons(n_samples=200, noise=0.15, random_state=0)
X = MinMaxScaler((0, np.pi)).fit_transform(X)
y = 2 * y - 1

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
print(X_train.shape, X_test.shape)

(140, 2) (60, 2)


## 2. Encoding the data

Angle encoding = one feature per qubit. 2 features -> 2 qubits.
(AmplitudeEmbedding fits 2^n features into n qubits, but the state prep circuit gets deep.)

In [3]:
n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def encode(x):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    return qml.state()

print(encode(X[0]))
print(qml.draw(encode)(X[0]))

[0.32614224+0.j 0.50068663+0.j 0.43764864+0.j 0.67186889+0.j]
0: ─╭AngleEmbedding(M0)─┤  State
1: ─╰AngleEmbedding(M0)─┤  State

M0 = 
[1.86072959 1.98688923]


## 3. The circuit

Encode, then a trainable layer, then measure Z on qubit 0.
BasicEntanglerLayers = one RX per qubit + a ring of CNOTs, repeated n_layers times.

In [4]:
n_layers = 3

@qml.qnode(dev)
def circuit(weights, x):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

shape = (n_layers, n_qubits)
weights = pnp.array(0.01 * np.random.randn(*shape), requires_grad=True)

print(qml.draw(circuit)(weights, X[0]))

0: ─╭AngleEmbedding(M0)─╭BasicEntanglerLayers(M1)─┤  <Z>
1: ─╰AngleEmbedding(M0)─╰BasicEntanglerLayers(M1)─┤     

M0 = 
[1.86072959 1.98688923]
M1 = 
[[ 0.0062163   0.00808659]
 [ 0.00586234 -0.00444115]
 [-0.01415624 -0.00375896]]


## 4. Training

Square loss + a classical bias term. Gradients come from the parameter-shift rule.
Passing X and y as keyword args so PennyLane doesn't try to differentiate them.

In [5]:
def predict(weights, bias, x):
    return circuit(weights, x) + bias

def cost(weights, bias, X, y):
    preds = pnp.stack([predict(weights, bias, x) for x in X])
    return pnp.mean((preds - y) ** 2)

def accuracy(weights, bias, X, y):
    preds = np.sign([predict(weights, bias, x) for x in X])
    return np.mean(preds == y)

bias = pnp.array(0.0, requires_grad=True)
opt = qml.NesterovMomentumOptimizer(0.2)

for i in range(30):
    idx = np.random.choice(len(X_train), 16, replace=False)
    weights, bias = opt.step(cost, weights, bias, X=X_train[idx], y=y_train[idx])

    if (i + 1) % 5 == 0:
        loss = cost(weights, bias, X_train, y_train)
        acc = accuracy(weights, bias, X_test, y_test)
        print(f"step {i+1}  loss {loss:.4f}  test acc {acc:.2f}")

step 5  loss 2.0700  test acc 0.32
step 10  loss 0.7814  test acc 0.67
step 15  loss 0.6964  test acc 0.68
step 20  loss 0.6916  test acc 0.65
step 25  loss 0.6961  test acc 0.65
step 30  loss 0.7512  test acc 0.68


## 5. Quantum kernel

Different approach — nothing trainable in the circuit. Encode x1, then apply the inverse
encoding of x2. The probability of measuring all-zeros is the overlap between the two states.
That's the kernel value, and sklearn's SVM takes it from there.

In [6]:
@qml.qnode(dev)
def kernel_circuit(x1, x2):
    qml.AngleEmbedding(x1, wires=range(n_qubits), rotation="Y")
    qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation="Y")
    return qml.probs(wires=range(n_qubits))

def kernel(x1, x2):
    return kernel_circuit(x1, x2)[0]

from sklearn.svm import SVC

K_train = qml.kernels.kernel_matrix(X_train, X_train, kernel)
K_test = qml.kernels.kernel_matrix(X_test, X_train, kernel)

svm = SVC(kernel="precomputed").fit(K_train, y_train)
print("kernel SVM test acc:", svm.score(K_test, y_test))

kernel SVM test acc: 0.85


## 6. Running it on Qiskit

Only the device line changes. qiskit.aer is a simulator with shot noise, so the value
won't match the exact one. Swapping in a real IBM backend works the same way.

In [7]:
dev_qiskit = qml.device("qiskit.aer", wires=n_qubits, shots=1024)

@qml.qnode(dev_qiskit)
def circuit_qiskit(weights, x):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

print("exact:", float(circuit(weights, X_test[0])))
print("1024 shots:", float(circuit_qiskit(weights, X_test[0])))

C:\Python314\Lib\site-packages\pennylane\devices\legacy_facade.py:212: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


exact: -0.10500358730817494
1024 shots: -0.126953125


## 7. Same thing in raw Qiskit

Building it by hand to see what PennyLane is doing underneath.
Pauli strings are little-endian in Qiskit, so Z on qubit 0 is written "IZ", not "ZI".

In [8]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator

x0, x1 = Parameter("x0"), Parameter("x1")
t0, t1 = Parameter("t0"), Parameter("t1")

qc = QuantumCircuit(2)
qc.ry(x0, 0)
qc.ry(x1, 1)
qc.cx(0, 1)
qc.ry(t0, 0)
qc.ry(t1, 1)

bound = qc.assign_parameters({x0: X_test[0][0], x1: X_test[0][1], t0: 0.3, t1: -0.2})
result = StatevectorEstimator().run([(bound, SparsePauliOp("IZ"))]).result()

print("<Z0> =", result[0].data.evs)
print(qc.draw())

<Z0> = -0.02660652771604477
     ┌────────┐     ┌────────┐
q_0: ┤ Ry(x0) ├──■──┤ Ry(t0) ├
     ├────────┤┌─┴─┐├────────┤
q_1: ┤ Ry(x1) ├┤ X ├┤ Ry(t1) ├
     └────────┘└───┘└────────┘


## 8. Making it hybrid

TorchLayer wraps the QNode as an nn.Module so it can sit inside a CNN.
Argument order matters — has to be (inputs, weights).
The Linear(64, 2) is standing in for CNN features for now.

In [9]:
import torch

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

qlayer = qml.qnn.TorchLayer(qnode, {"weights": shape})

model = torch.nn.Sequential(
    torch.nn.Linear(64, n_qubits),
    qlayer,
    torch.nn.Linear(n_qubits, 2),
)

print(model(torch.rand(4, 64)).shape)

torch.Size([4, 2])
